# Universal fd_cache generator (Kaggle GPU)

Generate FontDiffusion handwritten-style images for **every chu-Nom character** in the NomNaTong font CJK ranges (~21,800 chars). One-time run — produces a permanent cache that any future book can reuse without re-generating on Kaggle/Colab.

**Hardware**: Kaggle **GPU T4** (Settings → Accelerator → GPU T4 x2). P100 is faster but Kaggle's PyTorch 2.10 dropped Pascal (sm_60) support, so P100 fails. Time budget on T4: **~10-12 h, may need 2 sessions**.

**Resume-able**: every 500 chars, the notebook pushes the current state to a HuggingFace Hub repo. If Kaggle resets the session, just restart the notebook — cell 5 will re-download the partial cache and skip already-generated chars.

**Output**: `mdnt571/gannhanocr-universal-fd-cache` containing 21,837 PNGs named `U+XXXX.png`. Final size ~1.5 GB.

**Style strategy**: 1 representative crop from CacThanhTruyen2 — a clean, well-scanned book — used as the style reference for all chars. This produces a consistent style across the whole cache.

**Multi-font extension (Apr 2026):** the work universe is the UNION across NomNaTong + HanaMinA + HanaMinB + NotoSerifCJKsc (built by `build_char_universe.py`). Each character is rendered using the highest-priority font in the chain that contains it, so the cache now covers ~22,000+ CJK glyphs instead of just NomNaTong's ~21,800.

**Re-gen with tuned params (2026-05-18):** GUIDANCE_SCALE=3.0, ERODE_ITERS=1 (was 2.0/2). The old ERODE_ITERS=2 thinned strokes too aggressively (-50% ink vs source). New ERODE_ITERS=1 keeps ink ratio close to source font and real book crops. See `evaluation/test_fd_gen_params/` for the sweep results.

**Priority ordering (2026-05-18):** Chars used in the current 3-Sach dataset (~4,455 chars from `priority_chars.txt`) are generated FIRST. After session 1 (~3h), you can already pull cache + re-run the local pipeline. Remaining ~84k chars continue in later sessions for future-proofing.

## 1. Verify GPU + install deps

Check that GPU is enabled (notebook should already be set to GPU P100 in Settings).

Adds Kaggle-specific bits: secrets for `HF_TOKEN`, `tqdm` for progress.

In [ ]:
import shutil, sys
if shutil.which('nvidia-smi') is None:
    sys.exit('🛑 STOP: Runtime hiện tại không có GPU.\n'
             '   → Settings → Accelerator → GPU P100 → Save → Re-run all.')
!nvidia-smi

import subprocess
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
print(f'GPU detected: {gpu_name}')

# ── Pin PyTorch xuống version cuối cùng còn build sm_60 (Pascal P100) trong
#    binary wheel: torch 2.4.1+cu121.
#
#    Timeline drop sm_60:
#      • torch 2.4.x  cu121: arch list = sm_50;sm_60;sm_70;sm_75;sm_80;sm_86;sm_90  ✓
#      • torch 2.5.x  cu121: drop sm_50 + sm_60 → wheel chỉ sm_70+  ✗
#      • torch 2.6+        : index cu121 không còn, các index mới đều không sm_60
#
#    Nếu cài torch 2.5+ trên P100 sẽ load thành công nhưng tới khi gọi CUDA
#    kernel báo `cudaErrorNoKernelImageForDevice` → đây là gốc lỗi sanity test.
TORCH_PIN = 'torch==2.4.1+cu121'
TVISION_PIN = 'torchvision==0.19.1+cu121'

%pip install -q --force-reinstall --no-deps \
    {TORCH_PIN} {TVISION_PIN} \
    --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

%pip install -q \
    diffusers==0.36.0 \
    accelerate==1.12.0 \
    safetensors==0.7.0 \
    einops==0.8.1 \
    kornia==0.8.2 \
    info-nce-pytorch==0.1.4 \
    fonttools==4.61.1 \
    pygame==2.6.1 \
    scikit-image==0.26.0 \
    scipy==1.17.0 \
    huggingface-hub==0.36.0 \
    hf-xet==1.2.0 2>&1 | tail -3

# Gỡ peft/transformers — diffusers 0.36 import broken layerwise_casting helper từ peft.
%pip uninstall -y -q peft transformers 2>&1 | tail -2
print('✓ Removed peft/transformers (not needed by FontDiffusion).')

# ── Check CỨNG: torch trong RAM phải là 2.4.x VÀ arch list phải có sm_60.
#    Đây là 2 điều kiện độc lập — bản 2.5+ vẫn install sm_60 trong arch_list
#    cho gpu_capability check nhưng wheel KHÔNG có kernel sm_60 → fail runtime.
#    Vậy phải check bằng cách thử compile/load thực tế. Đơn giản nhất: pin 2.4.1
#    và check version + arch_list match.
import torch
ver = torch.__version__.split('+')[0]
if not ver.startswith('2.4.'):
    print(f'\n⚠️  Torch trong RAM là {torch.__version__}, không phải 2.4.x.')
    print('    Chạy: Run → Restart Session → Run all.')
    sys.exit('🛑 STOP: cần restart kernel để load torch 2.4.1.')

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    sm = f'sm_{cap[0]}{cap[1]}'
    ok = sm in supported
    print(f'  GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
    print(f'  Compute capability: {sm}  → {"✓ supported" if ok else "✗ NOT in supported list " + str(supported)}')
    if not ok:
        sys.exit(
            '\n🛑 STOP: GPU sm_60 không có trong arch list của torch hiện tại.\n'
            '   → Run → Restart Session → Run all (pip install vừa rồi cần restart để có hiệu lực).'
        )

    # Smoke test: dùng kernel CUDA thực sự để xác nhận wheel có sm_60 binary.
    try:
        _x = torch.randn(8, 8, device='cuda')
        _y = (_x @ _x).sum().item()
        print(f'  ✓ CUDA kernel smoke test passed (matmul on sm_60 OK)')
    except Exception as e:
        sys.exit(
            f'\n🛑 STOP: CUDA kernel fail trên sm_60: {type(e).__name__}: {e}\n'
            f'   → Wheel torch hiện tại thiếu sm_60 binary. Khả năng cao install bị skip\n'
            f'   do pip thấy torch đã có và bỏ qua (--force-reinstall không có hiệu lực).\n'
            f'   → Run → Restart Session, sau đó Run all (cell pip sẽ install sạch).'
        )

import diffusers
print(f'diffusers {diffusers.__version__}  (must be 0.36.0)')
assert diffusers.__version__ == '0.36.0', \
    f'diffusers version mismatch! Got {diffusers.__version__}, need 0.36.0. Restart kernel and re-run.'

try:
    import peft  # noqa: F401
    raise RuntimeError('peft is still installed — Restart kernel and re-run this cell.')
except ImportError:
    print('✓ peft absent → diffusers will skip the broken layerwise_casting import.')


## 2. Configure HuggingFace Hub

We push the partial cache to a HF Hub repo every 500 chars so a Kaggle session reset never wipes progress. Create the repo once (any name) and add `HF_TOKEN` to Kaggle Secrets (Add-ons → Secrets).

Edit `HF_REPO` below to your repo (will be created if missing).

In [ ]:
import os
from huggingface_hub import HfApi, login, create_repo

# ════════════════════════════════════════════════════════════════════
# Your HuggingFace Hub repo for the universal fd_cache.
# This will be auto-created if it doesn't exist.
# Format: '<hf-username>/<repo-name>'
# ════════════════════════════════════════════════════════════════════
HF_REPO = 'mdnt571/gannhanocr-fd-cache-v2'   # v2: ERODE=1 + FORCE_REGEN full 89k (separate from old v1 with ERODE=2)
HF_REPO_TYPE = 'dataset'                      # required by create_repo / list_repo_files below
# Token must be provided via env var or Kaggle Secrets — never hardcoded.
# On Kaggle:
#   Add-ons → Secrets → add a secret named HF_TOKEN with your HF write token.
# Locally:
#   export HF_TOKEN=hf_xxx  before launching jupyter.
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN not found — set env var or add Kaggle secret named HF_TOKEN.'
    )
# ════════════════════════════════════════════════════════════════════

login(token=HF_TOKEN, add_to_git_credential=False)

# Verify token + show identity
api = HfApi()
me = api.whoami()
print(f'✓ Logged in as: {me["name"]}')

expected_user = HF_REPO.split('/')[0]
if me['name'] != expected_user:
    print(f'⚠️  Token belongs to "{me["name"]}" but HF_REPO uses "{expected_user}".')
    print(f'    Either change HF_REPO to "{me["name"]}/..." or use a token from "{expected_user}".')

# Create dataset repo if missing
try:
    create_repo(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    print(f'✓ HF repo ready: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'⚠️  create_repo failed: {e}')
    print('   If your token is fine-grained, ensure it has:')
    print('     • Write access to repos under your username')
    print('     • "Create new repos" enabled')
    print('   Easiest fix: create a token of type "Write" instead of "Fine-grained".')

## 3. Clone the GanNhanOCR repo (with submodule)

Pulls main repo + `font_diffusion` submodule (FontDiffusion code + fonts).

In [3]:
import os, sys, shutil
from pathlib import Path

REPO_URL = 'https://github.com/truong571/GanNhanOCR.git'
PROJECT_ROOT = Path('/kaggle/working/GanNhanOCR')

def _populated(root: Path) -> bool:
    return (root / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf').exists()

if PROJECT_ROOT.exists():
    !cd {PROJECT_ROOT} && git pull --ff-only && git submodule update --init --recursive
    if not _populated(PROJECT_ROOT):
        # Stale empty submodule — wipe and re-init
        shutil.rmtree(PROJECT_ROOT / 'font_diffusion', ignore_errors=True)
        !cd {PROJECT_ROOT} && git submodule update --init --recursive
else:
    !git clone --recurse-submodules --depth 1 {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

assert _populated(PROJECT_ROOT), 'Submodule clone failed — fonts not present.'
print(f'✓ Repo at {PROJECT_ROOT}')
!ls font_diffusion/fonts/ | head -5

Cloning into '/kaggle/working/GanNhanOCR'...
remote: Enumerating objects: 123400, done.
remote: Counting objects: 100% (123400/123400), done.
remote: Compressing objects: 100% (122006/122006), done.
remote: Total 123400 (delta 1402), reused 123302 (delta 1388), pack-reused 0 (from 0)
Receiving objects: 100% (123400/123400), 319.86 MiB | 42.58 MiB/s, done.
Resolving deltas: 100% (1402/1402), done.
Updating files: 100% (123970/123970), done.
Submodule 'font_diffusion' (https://github.com/dzungphieuluuky/FontDiffusion.git) registered for path 'font_diffusion'
Cloning into '/kaggle/working/GanNhanOCR/font_diffusion'...
remote: Enumerating objects: 25434, done.        
remote: Counting objects: 100% (855/855), done.        
remote: Compressing objects: 100% (139/139), done.        
remote: Total 25434 (delta 786), reused 746 (delta 716), pack-reused 24579 (from 4)        
Receiving objects: 100% (25434/25434), 317.43 MiB | 35.92 MiB/s, done.
Resolving deltas: 100% (3928/3928), done.
Submodu

In [ ]:
# ── 3.5  Materialize medoid.png inside the cloned repo ─────────────────────
#
# medoid.png is the cosine-medoid of 120 chu-Nom crops produced by
# kaggle_diffusion/build_style_medoid.py on the Mac. It is .gitignored
# (*.png blanket rule) so `git clone` in cell 3 does NOT bring it. We embed
# the file inline as base64 so the notebook is self-contained — no Kaggle
# Dataset, no manual upload, survives session reset.
#
# To refresh after re-running build_style_medoid.py on the Mac, regenerate the
# blob with:
#   python3 -c "import base64,textwrap; print(textwrap.fill(base64.b64encode(open('kaggle_diffusion/style_references/medoid.png','rb').read()).decode(), 76))"
# and paste the output between the triple-quoted MEDOID_B64 below.

import base64

MEDOID_B64 = """
iVBORw0KGgoAAAANSUhEUgAAAHQAAABZCAIAAABgypeYAAAFd0lEQVR4Ae3BAYqsSgIAwcz7HzoX
BEFpq0e7Lf8ybyKs+DOHFX/msOLPHFb8mcOKP3NY8WcOK/7MYcWfOaz4M4cVf+aw4s8cVsyksqj4
x1gxk8qq4l9ixUwqRyp+OyvmUzlS8XtZ8QiVIxW/lBXzqQxU/FJWPELlSMWt1Ir/A1Y8QuVIxX1U
jlQ8zopHqBypuInKWxUPsuIRKkcqbqLyVsWDrHiEypGKm6i8VfEgKx6hcqTiPipjFQ+yYj6VIxW3
UhmreJAVk6m8qJhAZaziQVZMpvKiYgKVsYoHWTGTypGKCVROq5jJislUXlTcTWWvUhmrmMaKyVT2
KiZQ2ahYqRypmMaKmVRWFTOprCr2VF5UTGPFNCqriitU9irGVDYqXqgcqZjAimlUVhVjKj+pGFNZ
VYypvKi4mxXTqKwqBlROqBhQ2agYUzlScSsrplFZVbxQOa3iiMpGxQkqAxV3sGIOlVXFEZVzKsZU
VhUnqAxU3MGKCVT2KhYqF1WMqWxUvFArNlTGKr5mxa1UBiqViyreUtmo2FNZVGyoDFR8zYqbqFxX
saeyqnhLZaPiiMqiYkPlrYpPWXETlYsqXqisKsZU9iqOqCwq9lTGKj5lxXVqxZ7KORVjKouKMZW9
igGVVcWeyljFR6y4SGVRsaeyUbFQWVW8pbKoGFPZqBhQ2ah4oTJQ8RErLlJZVZyjsqgYU1lVjKls
VIyp7FWMqawqPmLFFSp7FeeoQMWAykbFmMqq4i2VFxUDKhsV11lxkcpexR1UVhVjKnsVAypHKsZU
9iqusOI6lb2KO6gsKsZU9ipA5bSKt1T2Kk6z4iMqexXfUVlVDKhsVCxUTqv4icqLinOs+IjKXsV3
VF5UbKhsVGyonFBxjsqLihOs+JTKquInKouKIypHKlYqGxUnqGxUnKNypOInVnxKZaPiLZVFxU9U
VhUrlVXFOSobFeeoHKn4iRWfUtmoeEtlUfETlUXFSmWj4hyVjYoTVMYq3rLiUyp7FWMqi4ojKlCp
rCpWKquK01Q2Kn6i8pOKMSu+oLJXMaCyqNhTGahYqawqTlPZqBhQuajiiBVfUHlRcURlUbGhMlCx
UtmoOE1lr2JD5QsVL6z4gsqLiiMqq4qFykDFSmWj4gqVr1WAypGKPSu+oHKk4oXKaRUbKquK61S+
ULFSOVKxYcUXVI5UvFA5rWJPZVFxncoJFeeovKhYWfEdlSMVGyqnVdxE5ZyKj6isKlZWfEflCxX3
UTmn4j4qi4qVFV9Tua7iPiqnVdxNBSpWVtxB5a1KZVExoLKoOE3lior5rLiJykAFqCwqBlRWFeeo
XFExnxW3UnlRASqLigGVtyqOqCwqXqhsVMxnxd1U9ipAZVExoPJWxXUqGxXzWTGBykYFqCwqBlTe
qrhOZaNiPisepLKoOKLyVsV1KhsV81nxIJVFxRGVtyquU9momM+KB6ksKo6ovFVxncpGxXxWPEhl
UfFC5YSKi1Q2Kuaz4kEqi4o9ldMqrlDZqJjPimepLCpAZaACVI5UnKayV3FEBSqVRcVHrHiWyqIC
VI5UrFQGKk5Q+ULFdVY8SOUnFS9UflIxoPKFiuuseJDKWxVjKv+pitOseJDKQMU5Kt+p2FM5p+Ic
Kx6hMlZxkco5lcpGxQuVEyrOsWI+lbGK76i8qNhQWVWMqbxVcYIVE6gsKpWxijuorCpeqKwqxlTe
qjjBivuoXFTxCJVVxVsqYxUnWHETlRMq/gsqGxVvqRypOMeKm6j8pOI/orJR8ROVFxXnWHETlUWl
sqhUFhX/EZW9isms+Aeo7FVMZsU/QGWvYjIrfjuVvYr5rPjtVPYq5rPit1PZq5jPin+AyqriEVb8
G1QWFY+w4s8c/wOooDSL4xsCTAAAAABJRU5ErkJggg==
"""

style_dir = PROJECT_ROOT / 'kaggle_diffusion' / 'style_references'
style_dir.mkdir(parents=True, exist_ok=True)
medoid_path = style_dir / 'medoid.png'
medoid_path.write_bytes(base64.b64decode(MEDOID_B64))
print(f'✓ medoid.png materialized at {medoid_path}  ({medoid_path.stat().st_size} bytes)')


## 4. Download FontDiffusion checkpoints from HF Hub

Pulls the 7 component files (unet, encoders, projections) into both DRO/ and FST/ checkpoint dirs so the loader finds them.

In [ ]:
from huggingface_hub import hf_hub_download

# Use the PRODUCTION weights at the HF repo root (fully trained, non-FST).
# The /FST/checkpoint_step_1500/ subfolder is a training snapshot at step 1500
# and produces near-noise output — do NOT use it.
# Production root only has unet/style_encoder/content_encoder; we run with
# use_fst=False (already set in core/ranking/fontdiffusion_gen.py).

FD_HF = 'dzungpham/font-diffusion-weights'
CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'PROD'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TO_FETCH = [
    'unet.safetensors',
    'style_encoder.safetensors',
    'content_encoder.safetensors',
]
for fn in TO_FETCH:
    dst = CKPT_DIR / fn
    if not dst.exists():
        cached = hf_hub_download(repo_id=FD_HF, filename=fn)  # ROOT, not FST/...
        shutil.copy2(cached, dst)
    print(f'  ✓ {fn}  ({dst.stat().st_size / 1024 / 1024:.1f} MB)')

# Wrapper still expects phase1_ckpt_dir for FST path; point it at the same dir
# (use_fst=False so it won\'t actually load FST modules).
FST_CKPT_DIR = CKPT_DIR

print(f'\n✓ FontDiffusion production weights ready at {CKPT_DIR}')


## 5. Resume from previous run (fast — list filenames only)

**Không** download các PNG đã có từ HF (sẽ tốn 30–60 phút × ~21k file → hay timeout
session Kaggle). Chỉ pull **danh sách filenames** (vài chục KB) qua `list_repo_files()`
để biết ký tự nào đã xong và skip.

Mac vẫn pull full cache cuối cùng qua `huggingface-cli download` (cell 22).

In [ ]:
import os as _os
# Tắt progress bar per-file của HF
_os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
from huggingface_hub import HfApi
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

CACHE_DIR = Path('/kaggle/working/_universal_fd_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# FAST RESUME: list_repo_files() chỉ pull danh sách filenames (vài chục KB) thay vì
# snapshot_download() pull TẤT CẢ PNG (~1.5 GB / ~21k file). Trên Kaggle mỗi
# session bị wipe /kaggle/working → snapshot_download lặp lại làm timeout session.
# Notebook chỉ cần biết "ký tự nào đã xong" để skip; bytes PNG không cần.
print(f'Listing existing PNGs on {HF_REPO} ...')
done_codepoints = set()
try:
    files = HfApi().list_repo_files(repo_id=HF_REPO, repo_type=HF_REPO_TYPE)
    for f in files:
        name = f.rsplit('/', 1)[-1]
        if name.startswith('U+') and name.endswith('.png'):
            done_codepoints.add(name[:-4])
except Exception as e:
    print(f'  (no remote cache yet: {type(e).__name__}: {str(e)[:120]})')

# Giữ tương thích với cell 13/19: 'existing' là list các local PNGs (rỗng cho session mới)
existing = sorted(CACHE_DIR.rglob('U+*.png'))
print(f'✓ Resume state: {len(done_codepoints):,} done on HF, {len(existing):,} local PNGs')


## 6. Build the work list

Read `kaggle_diffusion/exports/char_universe.txt` (21,837 chars). Filter out chars already in the cache. The remaining list is what this session needs to generate.

In [ ]:
# === Load FULL CJK universe (multi-font chain) ===
# char_universe.json is produced by build_char_universe.py and contains EVERY
# CJK glyph (Unified + Ext A-G + Compat) across the font chain:
#   NomNaTong + HanNomA/B + Han-Nom Kai/Minh + HanaMinA/B/C + Noto Serif CJK
# Typical size: ~89,000 chars. Each char has a `source_font` so we render it
# from the highest-priority font that actually contains the glyph.
universe_json = PROJECT_ROOT / 'kaggle_diffusion' / 'exports' / 'char_universe.json'
universe_txt  = PROJECT_ROOT / 'kaggle_diffusion' / 'exports' / 'char_universe.txt'

import json as _json
if universe_json.exists():
    meta = _json.loads(universe_json.read_text())
    all_chars    = [m['char'] for m in meta['chars']]
    char_to_font = {m['char']: m['source_font'] for m in meta['chars']}
    print(f"=== FULL CJK universe ===")
    print(f"Total chars: {meta['total_chars']:,}  ({len(meta['contributions_per_font'])} fonts)")
    for k, v in meta['contributions_per_font'].items():
        print(f"  {k:18s} {v:>6,}")
else:
    with open(universe_txt, encoding='utf-8') as f:
        all_chars = [line.strip() for line in f if line.strip()]
    char_to_font = {c: 'NomNaTong' for c in all_chars}
    print(f'Universe (legacy txt): {len(all_chars):,} chars, all NomNaTong')

# === FORCE_REGEN: ONLY needed when reusing an old HF repo with stale data. ==
# ==
# == Since we moved to a NEW HF repo (mdnt571/gannhanocr-fd-cache-v2 — empty  ==
# == at start), FORCE_REGEN MUST stay False so subsequent sessions skip the   ==
# == chars uploaded in earlier sessions. Otherwise session 2 would re-gen     ==
# == everything from session 1 = wasted GPU time.                             ==
# ==
# == Session 1: HF v2 empty → done_codepoints = 0 → todo = full 89k (priority ==
# ==            chars first), processes ~4500 chars in ~3h, uploads to v2.    ==
# == Session 2: HF v2 has 4500 → skips them → continues with rest 84k.        ==
FORCE_REGEN = False

done_codepoints |= {p.stem for p in existing}
universe_cps   = {f'U+{ord(c):04X}' for c in all_chars}
done_in_uni    = done_codepoints & universe_cps
extra_on_hf    = done_codepoints - universe_cps
if FORCE_REGEN:
    print('⚠  FORCE_REGEN=True — IGNORING HF cache, regenerating ALL chars')
    raw_todo = list(all_chars)
else:
    raw_todo = [c for c in all_chars if f'U+{ord(c):04X}' not in done_codepoints]
    print(f'Resume mode: {len(done_codepoints):,} already on HF, {len(raw_todo):,} remaining')

# === PRIORITY ORDERING ===
# Chars actually used in the current 3-Sach dataset → generate FIRST so the
# local pipeline can be re-run after just ~3h (4,455 chars × 2s on P100), then
# the remaining 84k chars finish in subsequent sessions for future-proofing.
priority_file = PROJECT_ROOT / 'kaggle_diffusion' / 'exports' / 'priority_chars.txt'
if priority_file.exists():
    priority_set = {ln.strip() for ln in priority_file.read_text(encoding='utf-8').splitlines() if ln.strip()}
    # Filter to chars that are in raw_todo AND in priority
    in_priority = [c for c in raw_todo if c in priority_set]
    in_rest     = [c for c in raw_todo if c not in priority_set]
    todo = in_priority + in_rest
    print(f'Priority order: {len(in_priority):,} corpus chars FIRST, then {len(in_rest):,} rest')
    print(f'   → After ~{len(in_priority)*2/3600:.1f}h on P100, you can pull cache + re-run local pipeline')
else:
    todo = raw_todo
    print(f'(no priority_chars.txt — running unsorted)')

print()
print(f"On HF:           {len(done_codepoints):>6,}  (+{len(extra_on_hf):,} ngoài universe)")
print(f"Already done:    {len(done_in_uni):>6,}")
print(f"==> TO GENERATE: {len(todo):>6,} chars")

# Group todo by source font — outer loop in cell 19 iterates these
from collections import Counter
todo_by_font: dict[str, list[str]] = {}
for c in todo:
    todo_by_font.setdefault(char_to_font[c], []).append(c)

if todo:
    print()
    print('TODO breakdown:')
    for fname, chars in todo_by_font.items():
        print(f'  {fname:18s} {len(chars):>6,}')
else:
    print('\n🎉 Universe cache complete!')


## 7. Load FontDiffusion pipeline (one time)

In [ ]:
# Per-font generator factory — call build_generator(font_name) to get a
# FontDiffusionGenerator pinned to that font's TTF. Used by cell 19 to switch
# fonts when iterating over `todo_by_font`.
import logging, warnings
logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

# Map FONT_CHAIN name -> file in font_diffusion/fonts/
# MUST match build_char_universe.py FONT_CHAIN exactly, or cell 19 will crash
# with 'Unknown font' when it hits a font_name not in this dict.
FONT_FILES = {
    # Vietnamese chu-Nom fonts (preferred — match historical manuscript style)
    'NomNaTong':      'NomNaTong-Regular.ttf',
    'NomNaTongLight': 'NomNaTongLight.ttf',
    'HanNomA':        'HAN NOM A.ttf',
    'HanNomB':        'HAN NOM B.ttf',
    'HanNomKai':      'Han-Nom-Khai-Regular-300623.ttf',
    'HanNomMinh':     'Han-nom Minh 1.42.otf',
    # Generic CJK backfill
    'HanaMinA':       'HanaMinA.ttf',
    'HanaMinB':       'HanaMinB.ttf',
    'HanaMinC':       'HanaMinC.otf',
    # Optional final backstop
    'NotoSerifCJKsc': 'NotoSerifCJKsc-Regular.otf',
}

_GEN_CACHE = {}
_SKIP_FONTS: set = set()   # fonts we've already failed to load — skip silently

def build_generator(font_name: str) -> 'FontDiffusionGenerator | None':
    """Return a generator for `font_name`, or None if its TTF is missing on disk.
    Caller (cell 19) should skip chars assigned to that font when None."""
    if font_name in _GEN_CACHE:
        return _GEN_CACHE[font_name]
    if font_name in _SKIP_FONTS:
        return None
    if font_name not in FONT_FILES:
        print(f'⚠  Unknown font {font_name!r} — not in FONT_FILES, skipping')
        _SKIP_FONTS.add(font_name)
        return None
    fpath = PROJECT_ROOT / 'font_diffusion' / 'fonts' / FONT_FILES[font_name]
    if not fpath.exists():
        print(f'⚠  Font file missing: {fpath.name} — skipping all {font_name} chars')
        _SKIP_FONTS.add(font_name)
        return None
    print(f'Loading FontDiffusion with {font_name} ({fpath.name})...')
    g = FontDiffusionGenerator(
        ckpt_dir=str(CKPT_DIR),
        phase1_ckpt_dir=str(FST_CKPT_DIR),
        font_path=str(fpath),
        cache_dir=str(CACHE_DIR),
    )
    _GEN_CACHE[font_name] = g
    print(f'✓ Loaded ({font_name})')
    return g

# Pre-load the primary font so cell 17 sanity test works out of the box.
generator = build_generator('NomNaTong')


## 7.5 Sanity test — generate 5 sample chars

Before committing to a 6-8 h run, generate 5 representative chars and verify the
output looks reasonable. This is fast (~30 s) and catches:
- Broken style image (empty / wrong format)
- FontDiffusion config mismatch
- GPU OOM at this batch size
- Per-char generator errors

If the output thumbnails look like handwritten chu-Nom (not noise / blank), proceed
to cell 8. Otherwise, fix the issue and re-run this cell.

In [ ]:
import os
# ── Phải set TRƯỚC khi pygame/SDL được import bởi font_diffusion ──
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')      # tắt ALSA errors
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')      # headless safe
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', 'hide')  # tắt banner pygame
os.environ.setdefault('XDG_RUNTIME_DIR', '/tmp')        # tắt 'XDG_RUNTIME_DIR not set'
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')
os.environ.setdefault('DIFFUSERS_VERBOSITY', 'error')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')

import time, logging, warnings, io, sys
from contextlib import redirect_stdout, redirect_stderr
import numpy as np
import cv2
from PIL import Image
from IPython.display import display

# Tắt mọi logger noisy của FontDiffusion + deps; disable propagation để khỏi lên root.
for _name in ('src', 'src.builders', 'src.builders.build', 'src.model', 'src.modules',
              'diffusers', 'accelerate', 'huggingface_hub', 'transformers'):
    _lg = logging.getLogger(_name)
    _lg.setLevel(logging.ERROR)
    _lg.propagate = False
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

try:
    import diffusers
    diffusers.utils.logging.set_verbosity_error()
    if hasattr(diffusers.utils.logging, 'disable_progress_bar'):
        diffusers.utils.logging.disable_progress_bar()
except Exception:
    pass


# ── _Discard (memory-safe) thay cho io.StringIO trong redirect ──
class _Discard:
    def write(self, *a, **kw): return 0
    def flush(self): pass
    def isatty(self): return False
    def writable(self): return True
    def readable(self): return False
    def seekable(self): return False
    def close(self): pass
    @property
    def closed(self): return False
_devnull = _Discard()


class _SilenceCtx:
    """Redirect cả stdout + stderr → _devnull (không tích RAM)."""
    def __enter__(self):
        self._so = redirect_stdout(_devnull); self._so.__enter__()
        self._se = redirect_stderr(_devnull); self._se.__enter__()
        return self
    def __exit__(self, *a):
        self._se.__exit__(*a); self._so.__exit__(*a)


# Tuned params (verified via evaluation/test_fd_gen_params/sweep on 2026-05-18):
#   - GUIDANCE_SCALE 3.0 → +0.027 DINOv2 cosine vs source (free upgrade, same time)
#   - ERODE_ITERS  1   → ink ratio 13.1% (closest to source font 10.8%, real crop 7.6%)
#                         ERODE=2 (old) produced 5.4% ink → over-thinning, dropped strokes
#   - num_inference_steps stays at 20 — bumping to 40-60 only gave +0.03 cos for 2-3× time
STYLE_NAME_ID = 'medoid'
GUIDANCE_SCALE = 3.0
ERODE_ITERS = 1

STYLE_IMAGE_PATH = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_NAME_ID}.png')
assert Path(STYLE_IMAGE_PATH).exists(), f'Style image missing: {STYLE_IMAGE_PATH}'
print(f'Style: {STYLE_IMAGE_PATH}')
display(Image.open(STYLE_IMAGE_PATH).resize((128, 128)))

# Silence src.builders.build/src.model prints + pygame/ALSA noise khi build pipeline
print('Loading FontDiffusion pipeline (silenced)...', flush=True)
_t_load = time.time()
try:
    with _SilenceCtx():
        generator._load_pipeline()
except Exception as e:
    print(f'✗ _load_pipeline failed: {type(e).__name__}: {e}')
    raise
print(f'✓ pipeline loaded in {time.time()-_t_load:.1f}s')

generator.pipe.guidance_scale = GUIDANCE_SCALE
if hasattr(generator.pipe, 'set_progress_bar_config'):
    try:
        generator.pipe.set_progress_bar_config(disable=True)
    except Exception:
        pass
print(f'guidance_scale = {generator.pipe.guidance_scale}  |  erode_iters = {ERODE_ITERS}')


def erode_strokes(img_path, iters):
    try:
        arr = np.array(Image.open(img_path).convert('L'))
        if iters > 0:
            arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=iters)
        Image.fromarray(arr).save(img_path)
    except Exception as e:
        print(f'  ⚠️  erode {img_path.name}: {type(e).__name__}: {e}')


TEST_CHARS = ['一', '二', '三', '人', '月']
print(f'\nGenerating {len(TEST_CHARS)} sanity-test chars: {TEST_CHARS}', flush=True)

t0 = time.time()
try:
    with _SilenceCtx():
        generator.generate(TEST_CHARS, STYLE_IMAGE_PATH, style_name='_sanity')
    print(f'  Time: {(time.time()-t0):.1f}s ({(time.time()-t0)/len(TEST_CHARS):.1f}s/char)')
except Exception as e:
    print(f'  ✗ generate failed: {type(e).__name__}: {e}')
    raise

sanity_dir = CACHE_DIR / '_sanity'
for c in TEST_CHARS:
    p = sanity_dir / f'U+{ord(c):04X}.png'
    if p.exists():
        erode_strokes(p, ERODE_ITERS)

print('\nGenerated images (post-erosion):')
for c in TEST_CHARS:
    img_path = sanity_dir / f'U+{ord(c):04X}.png'
    if img_path.exists():
        print(f'  ✓ {c} → U+{ord(c):04X}.png')
        display(Image.open(img_path).resize((96, 96)))
    else:
        print(f'  ✗ {c} → not generated')

import shutil as _sh
if sanity_dir.exists():
    _sh.rmtree(sanity_dir, ignore_errors=True)
print('\n✓ Sanity test complete. Inspect images above. If they look like handwritten chu-Nom, proceed.')


## 8. Generate + resilient checkpoint upload (Kaggle-tuned, quiet)

Vòng lặp: sinh theo chunk (`CHUNK=2000`), watcher chạy nền flatten + push HF Hub mỗi khi đủ ~500 file mới hoặc ≥30 phút từ lần upload thành công gần nhất.

**Tinh chỉnh giảm log để Kaggle UI khỏi lag (full run ~21k chars × 10–14h):**
- `_Discard` class (không phải `io.StringIO`) nuốt mọi log per-batch của FontDiffusion → không tích RAM.
- Redirect cả stdout VÀ stderr → diffusers' per-step tqdm cũng bị nuốt.
- `set_progress_bar_config(disable=True)` trên pipeline + `disable_progress_bar()` trên diffusers.
- `tqdm(mininterval=30, miniters=200)` → main bar refresh tối đa 1 lần/30s.
- `print_report=False` cho `upload_large_folder` → không in báo cáo upload định kỳ.
- Bỏ các print verbose (watcher started/stopped chi tiết, "chunk already present", "flattened N extra").

**Tinh chỉnh checkpoint cho Kaggle full run:**
- `CHUNK=2000` (↑ từ 200) → ít commit hơn (tránh HF 429 commit-per-hour).
- Watcher: chờ ≥500 file mới hoặc ≥30 phút mới push.
- `safe_generate()` bắt riêng `torch.cuda.OutOfMemoryError` → fallback per-char khi OOM.

In [ ]:
import os, time, threading, traceback, logging, warnings, io, sys
from contextlib import redirect_stdout, redirect_stderr
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
import huggingface_hub
from huggingface_hub.utils import disable_progress_bars

# ── Quiet mode: tắt mọi progress bar + log không thiết yếu ──
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['DIFFUSERS_VERBOSITY'] = 'error'
disable_progress_bars()
logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('src.model').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub._upload_large_folder').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')

# Tắt progress bar nội bộ của diffusers (per-step inference bar in stderr)
try:
    import diffusers
    diffusers.utils.logging.set_verbosity_error()
    if hasattr(diffusers.utils.logging, 'disable_progress_bar'):
        diffusers.utils.logging.disable_progress_bar()
except Exception:
    pass

# Kaggle GPU (P100/T4 16 GB VRAM): batch_size=4 mặc định OK.
# Full run ~21k chars trên P100: ~10–14h → cần chunk dày + watcher upload nền.
CHUNK = 2000                       # ↑ từ 200 — ít upload cycle
UPLOAD_NUM_WORKERS = 4
UPLOAD_RETRIES = 4
UPLOAD_BACKOFF_SECONDS = 60
WATCHER_INTERVAL_SEC = 120
WATCHER_MIN_FILES = 2000           # ↑ từ 500 — giảm số commit/giờ
MAX_UPLOAD_GAP_SEC = 3600          # ↑ từ 1800 — 1 giờ max gap
FILE_STABLE_AGE_SEC = 2
# === Tuned params (see cell 17 for rationale) ===
STYLE_NAME_ID = 'medoid'
GUIDANCE_SCALE = 3.0
ERODE_ITERS = 1
STYLE_NAME = 'universal'
STYLE_IMAGE = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_NAME_ID}.png')

assert Path(STYLE_IMAGE).exists(), f'Style image missing: {STYLE_IMAGE}'
if not hasattr(api, 'upload_large_folder'):
    raise RuntimeError(
        f'huggingface_hub {huggingface_hub.__version__} lacks upload_large_folder(). '
        'Re-run cell 1, then restart kernel.'
    )

generator._load_pipeline()
generator.pipe.guidance_scale = GUIDANCE_SCALE
if hasattr(generator.pipe, 'set_progress_bar_config'):
    try:
        generator.pipe.set_progress_bar_config(disable=True)
    except Exception:
        pass
print(f'CONFIG: style={STYLE_NAME_ID}  guidance={GUIDANCE_SCALE}  chunk={CHUNK}  '
      f'watcher={WATCHER_INTERVAL_SEC}s')


def _erode_one(p):
    try:
        arr = np.array(Image.open(p).convert('L'))
        if ERODE_ITERS > 0:
            arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=ERODE_ITERS)
        Image.fromarray(arr).save(p)
    except Exception:
        pass


def _bucket_dir(codepoint_hex: str) -> str:
    """Bucket subdir name for a codepoint hex string.
    First 2 hex chars → 256 buckets, max ~200 files/bucket cho full universe.
    Tránh HF 10000-files-per-directory limit."""
    return codepoint_hex[:2]


def _local_path(char: str) -> Path:
    """Đường dẫn local của PNG cho 1 char (bucket subdir OR legacy root)."""
    cp_hex = f'{ord(char):04X}'
    bucket_path = CACHE_DIR / _bucket_dir(cp_hex) / f'U+{cp_hex}.png'
    legacy_path = CACHE_DIR / f'U+{cp_hex}.png'   # ~9,995 PNG cũ ở root
    return bucket_path if bucket_path.exists() else legacy_path


sub = CACHE_DIR / STYLE_NAME
sub.mkdir(parents=True, exist_ok=True)

_flatten_lock = threading.Lock()

# ── CRITICAL: KHÔNG dùng io.StringIO() — nó tích lũy mọi stdout bị redirect trong
# RAM, sau 10–14h × hàng nghìn batch log của FontDiffusion → tràn RAM Kaggle (13 GB).
# _Discard.write() bỏ ngay, không giữ tham chiếu.
class _Discard:
    def write(self, *a, **kw): return 0
    def flush(self): pass
    def isatty(self): return False
    def writable(self): return True
    def readable(self): return False
    def seekable(self): return False
    def close(self): pass
    @property
    def closed(self): return False
_devnull = _Discard()


class _SilenceCtx:
    """Redirect cả stdout + stderr vào _devnull (memory-safe)."""
    def __enter__(self):
        self._so = redirect_stdout(_devnull); self._so.__enter__()
        self._se = redirect_stderr(_devnull); self._se.__enter__()
        return self
    def __exit__(self, *a):
        self._se.__exit__(*a)
        self._so.__exit__(*a)
def _silence(): return _SilenceCtx()


def flatten(require_stable=True):
    """Move PNGs từ sub/universal/ → CACHE_DIR/<bucket>/ (bucket = first 2 hex chars).
    PNG đã có ở bucket HOẶC legacy root đều skip + unlink bản ở sub."""
    moved = []
    now = time.time()
    with _flatten_lock:
        for p in list(sub.glob('U+*.png')):
            try:
                if require_stable and (now - p.stat().st_mtime) < FILE_STABLE_AGE_SEC:
                    continue
                cp_hex = p.stem[2:]   # strip 'U+', e.g. '4E8C'
                bucket = CACHE_DIR / _bucket_dir(cp_hex)
                bucket.mkdir(exist_ok=True)
                dst = bucket / p.name
                legacy = CACHE_DIR / p.name   # PNG cũ từ resume trước đây
                if dst.exists() or legacy.exists():
                    p.unlink()
                else:
                    _erode_one(p)
                    shutil.move(str(p), str(dst))
                    moved.append(dst)
            except (FileNotFoundError, Exception):
                continue
    return moved


_upload_lock = threading.Lock()
_upload_state = {'thread': None, 'last_success': time.time()}


def _upload_blocking(label):
    n = sum(1 for _ in CACHE_DIR.rglob('U+*.png'))
    if n == 0:
        return True
    for attempt in range(1, UPLOAD_RETRIES + 1):
        try:
            with _silence():
                api.upload_large_folder(
                    repo_id=HF_REPO,
                    repo_type=HF_REPO_TYPE,
                    folder_path=str(CACHE_DIR),
                    allow_patterns=['U+*.png', '*/U+*.png'],
                    ignore_patterns=['universal/*'],   # staging dir — generator đang ghi
                    num_workers=UPLOAD_NUM_WORKERS,
                    print_report=False,
                )
            _upload_state['last_success'] = time.time()
            print(f'  ✓ [{label}] pushed (HF={n:,})', flush=True)
            return True
        except Exception as e:
            wait = UPLOAD_BACKOFF_SECONDS * attempt
            msg = str(e)[:120].replace(chr(10), ' ')
            print(f'  ⚠️  [{label}] try {attempt}/{UPLOAD_RETRIES}: {type(e).__name__}: {msg}', flush=True)
            if attempt == UPLOAD_RETRIES:
                print(f'  ⚠️  [{label}] giving up — local intact, retry next checkpoint.', flush=True)
                return False
            time.sleep(wait)
    return False


def _upload_target(label):
    with _upload_lock:
        try:
            _upload_blocking(label)
        except Exception as e:
            print(f'  ⚠️  [{label}] bg upload crashed: {type(e).__name__}', flush=True)


def maybe_upload_async(label):
    prev = _upload_state['thread']
    if prev is not None and prev.is_alive():
        return False
    t = threading.Thread(target=_upload_target, args=(label,), daemon=True)
    t.start()
    _upload_state['thread'] = t
    return True


def upload_wait():
    t = _upload_state['thread']
    if t is not None and t.is_alive():
        print('  ⏳ waiting for upload...', flush=True)
        t.join()


_watcher_state = {'stop': False, 'thread': None, 'pending': 0}


def _watcher_loop():
    while not _watcher_state['stop']:
        try:
            moved = flatten(require_stable=True)
            if moved:
                _watcher_state['pending'] += len(moved)
            prev = _upload_state['thread']
            idle = prev is None or not prev.is_alive()
            pending = _watcher_state['pending']
            gap = time.time() - _upload_state['last_success']
            if idle and pending > 0 and (pending >= WATCHER_MIN_FILES or gap >= MAX_UPLOAD_GAP_SEC):
                reason = f'+{pending}' if pending >= WATCHER_MIN_FILES else f'+{pending}/gap{int(gap)}s'
                _watcher_state['pending'] = 0
                maybe_upload_async(label=f'wch{reason}')
        except Exception:
            pass
        for _ in range(WATCHER_INTERVAL_SEC):
            if _watcher_state['stop']:
                return
            time.sleep(1)


def start_watcher():
    _watcher_state['stop'] = False
    _watcher_state['pending'] = 0
    _upload_state['last_success'] = time.time()
    t = threading.Thread(target=_watcher_loop, daemon=True)
    t.start()
    _watcher_state['thread'] = t
    # watcher started silently


def stop_watcher():
    _watcher_state['stop'] = True
    t = _watcher_state['thread']
    if t is not None:
        t.join(timeout=WATCHER_INTERVAL_SEC + 5)
    _watcher_state['thread'] = None


def safe_generate(gen, batch):
    """gen.generate() spams per-batch logs — _silence() drops stdout+stderr.
    OOM fallback to per-char. Errors collected, never printed unless >10."""
    failed = []
    try:
        with _silence():
            gen.generate(batch, STYLE_IMAGE, style_name=STYLE_NAME)
        return failed
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        for c in batch:
            try:
                with _silence():
                    gen.generate([c], STYLE_IMAGE, style_name=STYLE_NAME)
            except Exception as ce:
                failed.append((c, f'{type(ce).__name__}'))
                torch.cuda.empty_cache()
        return failed
    except Exception as e:
        torch.cuda.empty_cache()
        for c in batch:
            try:
                with _silence():
                    gen.generate([c], STYLE_IMAGE, style_name=STYLE_NAME)
            except Exception as ce:
                failed.append((c, f'{type(ce).__name__}'))
                torch.cuda.empty_cache()
        return failed


for _font_name, _font_todo in todo_by_font.items():
    if not _font_todo:
        continue
    print(f'\n=== Generating {len(_font_todo):,} chars using {_font_name} ===')
    current_gen = build_generator(_font_name)
    if current_gen is None:
        print(f'   ↳ skipped: no generator for {_font_name} ({len(_font_todo):,} chars deferred)')
        continue
    todo = _font_todo  # cell-19 body iterates over `todo`
    if todo:
        t0 = time.time()
        pbar = tqdm(total=len(todo), desc='Generate', unit='char',
                    dynamic_ncols=True, mininterval=30.0, miniters=200)
        all_failed = []
        start_watcher()
        try:
            for i in range(0, len(todo), CHUNK):
                batch = todo[i:i + CHUNK]
                missing_batch = [c for c in batch if not _local_path(c).exists()]
    
                if missing_batch:
                    all_failed.extend(safe_generate(current_gen, missing_batch))
    
                moved = flatten(require_stable=True)
                if moved:
                    _watcher_state['pending'] += len(moved)
    
                pbar.update(len(batch))
                # Watcher tự lo upload — không trigger từ main loop (tránh 429).
                torch.cuda.empty_cache()
        except KeyboardInterrupt:
            print('\n⚠️  interrupted — letting in-flight upload finish.')
        except Exception as e:
            print(f'\n⚠️  loop error: {type(e).__name__}: {e}')
            traceback.print_exc()
        finally:
            pbar.close()
            stop_watcher()
            leftover = flatten(require_stable=False)
            if leftover:
                print(f'  flattened {len(leftover):,} leftover', flush=True)
            upload_wait()
            _upload_blocking(label='final')
            if all_failed:
                print(f'\n⚠️  {len(all_failed)} chars failed (first 5):')
                for c, err in all_failed[:5]:
                    print(f'    {c} (U+{ord(c):04X}): {err}')
            print(f'\n✓ done in {(time.time()-t0)/60:.1f} min')
    else:
        print('Nothing to generate.')
    

## 9. Final state + verification

After generation finishes (or session resets), confirm cache size matches universe.

In [ ]:
# Final state check — đọc từ HF (notebook không còn download local).
from huggingface_hub import HfApi
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

remote_files = HfApi().list_repo_files(repo_id=HF_REPO, repo_type=HF_REPO_TYPE)
remote_cps = set()
for f in remote_files:
    name = f.rsplit('/', 1)[-1]
    if name.startswith('U+') and name.endswith('.png'):
        remote_cps.add(name[:-4])

universe_cps = {f'U+{ord(c):04X}' for c in all_chars}
done_in_universe = remote_cps & universe_cps
missing = universe_cps - remote_cps
extra = remote_cps - universe_cps

print(f'HF repo: {HF_REPO}')
print(f'  Total PNGs on HF:  {len(remote_cps):>6,}')
print(f'  In current universe: {len(done_in_universe):>6,} / {len(universe_cps):,}')
print(f'  Extra (PUA/Ext-G):   {len(extra):>6,}')
print(f'  Coverage (universe): {len(done_in_universe)/len(universe_cps)*100:.2f}%')

# Local count (sẽ là 0 trong session mới — không phải bug)
local = sorted(CACHE_DIR.rglob('U+*.png'))
local_size_mb = sum(p.stat().st_size for p in local) / 1024 / 1024
print(f'\nLocal cache (chỉ trong session này): {len(local):,} PNGs ({local_size_mb:.0f} MB)')
print('  ↑ rỗng là bình thường — Mac sẽ pull full từ HF qua huggingface-cli download.')

if not missing:
    print('\n🎉 Universe HOÀN CHỈNH trên HF — sẵn sàng pull về Mac.')
else:
    print(f'\n⚠️  Còn {len(missing):,} chars thiếu trên HF. Restart notebook để sinh tiếp.')
    print(f'   Mẫu thiếu: {sorted(missing)[:5]}')


## 10. Use the cache from your Mac

Once 100% complete, on the Mac side:

```sh
cd /path/to/GanNhanOCR
huggingface-cli download mdnt571/gannhanocr-universal-fd-cache \
  --repo-type=dataset \
  --local-dir prepared/_universal_fd_cache/

./run_pipeline.sh --step 3 && ./run_pipeline.sh --step 4
```

Step 3 will use the universal cache for any chu-Nom char in any book — no more Colab/Kaggle generation runs needed.